In [1]:
import torch
import sys 
from pathlib import Path
src_path=Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
import DeepGPR
import matplotlib.pyplot as plt

# try:
#     assert torch.cuda.is_available()
#     print(f"CUDA is available. Using device: {torch.cuda.get_device_name(0)}")
#     device = torch.device("cuda")
# except AssertionError:
#     raise RuntimeError("CUDA is not available. Please check your PyTorch installation and GPU setup.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set up the parameters and models
dx=0.02
dt=3e-11
nt=2000
er = torch.ones(100, 100,1) * 2  
er[50:,:]=5
se = torch.zeros_like(er)  
er.requires_grad_()
source_location=torch.tensor([[[10,10,0]]],device=device,dtype=torch.int)
receiver_location=torch.tensor([[[10,90,0]]],device=device,dtype=torch.int)
freq=2e8
peak_time = 1 / freq
source_amplitudes = torch.zeros((1,nt,1),device=device)
source_amplitudes[0,:,0]=DeepGPR.ricker(freq, nt, dt, peak_time).to(device)

#forward modeling
r = DeepGPR.compute(
    device=device, dx=dx, dt=dt, 
    source_amplitudes=source_amplitudes,
    source_location=source_location, 
    receiver_location=receiver_location, 
    er=er, se=se
)

(r[-1]**2).sum().backward()

_, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(r[-1].detach().flatten().cpu().numpy())
ax[0].set_title("Receiver data")
ax[1].imshow(er.grad.detach())
ax[1].set_title("Gradient")
plt.show()


FileNotFoundError: DeepGPR CPU shared library was not found or could not be loaded.

Current platform: Darwin
Expected one of:
/Users/llsra/Desktop/DeepGPR/src/DeepGPR/lib/libdeepgpr_cpu.dylib
/Users/llsra/Desktop/DeepGPR/src/DeepGPR/lib/deepgpr_cpu.dylib

Available files in /Users/llsra/Desktop/DeepGPR/src/DeepGPR/lib:
['.DS_Store', 'deepgpr.cu', 'deepgpr.dll', 'deepgpr.so', 'deepgpr_cpu.c', 'deepgpr_cpu.dll', 'deepgpr_cpu.dylib', 'deepgpr_cpu.so']

Load details:
/Users/llsra/Desktop/DeepGPR/src/DeepGPR/lib/deepgpr_cpu.dylib: dlopen(/Users/llsra/Desktop/DeepGPR/src/DeepGPR/lib/deepgpr_cpu.dylib, 0x0006): Library not loaded: /opt/homebrew/opt/libomp/lib/libomp.dylib
  Referenced from: <A06EF6EF-1384-3F43-90C4-4718DEC6E8AF> /Users/llsra/Desktop/DeepGPR/src/DeepGPR/lib/deepgpr_cpu.dylib
  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/l